# Precision vs Recall — Threshold Demo

### Simple FDE use case: suspicious insurance claims

Imagine a model gives every claim a **risk score from 0 to 100**.

- Higher score = more suspicious
- Score >= threshold → send to the review queue
- Score < threshold → straight-through path

**Learning objective:** Move the threshold and observe how **precision, recall, false positives and false negatives** change.


## The story

There are 20 claims. We already know the historical outcome for this small teaching dataset:

- 🟥 = actual fraud
- 🟦 = legitimate claim

> **Where should we set the threshold so the Risk team catches enough fraud without creating an unmanageable review queue?**


In [ ]:
import pandas as pd

data = [
    (95,1),(92,1),(88,1),(84,1),(79,1),(73,1),(68,1),(61,1),(55,1),(48,1),
    (90,0),(82,0),(76,0),(70,0),(64,0),(58,0),(50,0),(42,0),(30,0),(15,0)
]
df = pd.DataFrame(data, columns=['risk_score','fraud'])
df['actual'] = df['fraud'].map({1:'Fraud', 0:'Legitimate'})
df = df.sort_values('risk_score', ascending=False).reset_index(drop=True)
df


## 1. What does a threshold do?

Suppose we choose **70**:

```text
risk score >= 70  →  REVIEW
risk score <  70  →  STRAIGHT THROUGH
```

The threshold is therefore a **business control**: it determines how many cases enter the queue and how much risk may slip through.


In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score

def evaluate(threshold):
    predicted = (df['risk_score'] >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(df['fraud'], predicted).ravel()
    precision = precision_score(df['fraud'], predicted, zero_division=0)
    recall = recall_score(df['fraud'], predicted, zero_division=0)
    return {'threshold':threshold,'queue':tp+fp,'tp':tp,'fp':fp,'fn':fn,'tn':tn,'precision':precision,'recall':recall}

r = evaluate(70)
print(f"Threshold: {r['threshold']}")
print(f"Review queue: {r['queue']} claims")
print(f"Fraud caught (TP): {r['tp']}")
print(f"False alarms (FP): {r['fp']}")
print(f"Fraud missed (FN): {r['fn']}")
print(f"Precision: {r['precision']:.1%}")
print(f"Recall:    {r['recall']:.1%}")


## 2. The FDE interpretation

**Precision asks Operations:** “Of the cases we put in the queue, how many were actually fraud?”

**Recall asks Risk:** “Of all the fraud that existed, how much did we catch?”

Think of precision as **queue usefulness** and recall as **risk coverage**.


In [ ]:
results = []
for threshold in [90,80,70,60,50,40,30,20,15]:
    r = evaluate(threshold)
    results.append({
        'Threshold': threshold,
        'Review queue': r['queue'],
        'Fraud caught': r['tp'],
        'False alarms': r['fp'],
        'Fraud missed': r['fn'],
        'Precision': round(r['precision'],3),
        'Recall': round(r['recall'],3)
    })
pd.DataFrame(results)


## 3. Interactive threshold slider

Move the threshold from **95 down to 15**. Notice the trade-off.

- Lower threshold → more cases reviewed → usually higher recall
- Higher threshold → fewer cases reviewed → usually higher precision

Neither direction is automatically “better.” The business must decide the acceptable trade-off.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

slider = widgets.IntSlider(value=70, min=15, max=95, step=5, description='Threshold:', continuous_update=False)
out = widgets.Output()

def show(change=None):
    r = evaluate(slider.value)
    with out:
        clear_output(wait=True)
        print(f"Threshold = {r['threshold']}")
        print(f"Review queue       : {r['queue']} claims")
        print(f"Fraud caught       : {r['tp']}")
        print(f"False alarms       : {r['fp']}")
        print(f"Fraud missed       : {r['fn']}")
        print(f"Precision           : {r['precision']:.1%}")
        print(f"Recall              : {r['recall']:.1%}")
        print('\nOperations → How useful is the queue?')
        print('Risk       → How much of the real fraud did we catch?')

slider.observe(show, names='value')
display(slider, out)
show()


In [ ]:
import matplotlib.pyplot as plt

thresholds = list(range(15,100,5))
rows = [evaluate(t) for t in thresholds]
precision_values = [r['precision'] for r in rows]
recall_values = [r['recall'] for r in rows]

plt.figure(figsize=(9,5))
plt.plot(thresholds, precision_values, marker='o', label='Precision')
plt.plot(thresholds, recall_values, marker='o', label='Recall')
plt.xlabel('Risk threshold')
plt.ylabel('Metric')
plt.title('Threshold vs Precision and Recall')
plt.ylim(0,1.05)
plt.grid(True, alpha=0.25)
plt.legend()
plt.show()


## 4. FDE exercise — make the business decision

Assume the real system processes **4,000 claims/month (~133/day)** and the investigation team can comfortably handle **15 additional reviews/day**.

### Team task

1. Pick a threshold.
2. Estimate the review queue.
3. State precision and recall.
4. Explain how many fraud cases may still be missed.
5. Defend the threshold to both Operations and Risk.
6. Now imagine the review team doubles. Would you change the threshold?

> **Precision tells Operations how useful the queue is. Recall tells Risk how much of the real risk we caught. The threshold is where those two concerns meet.**


## Trainer discussion prompt

Don't ask only: **“Which threshold gives the best precision?”**

Ask instead:

> **“What level of missed fraud are we willing to tolerate, and how much investigation capacity are we willing to spend to reduce it?”**

That turns an ML metric into an FDE/business conversation.
